#### import dependencies

In [1]:
from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import Filter, FieldCondition, MatchValue, Prefetch, FusionQuery

import psycopg2
from psycopg2.extras import RealDictCursor
import numpy as np
import os

#### Add to shopping cart tool

In [3]:
items = [
    {
        "product_id": "B0BR8SJM7N",
        "quantity": 2
    },
    {
        "product_id": "B09JS5PSV1",
        "quantity": 4
    }
]

In [4]:
def add_to_shopping_cart(items: list[dict], user_id: str, cart_id: str) -> str:

    """Add a list of provided items to the shopping cart.
    
    Args:
        items: A list of items to add to the shopping cart. Each item is a dictionary with the following keys: product_id, quantity.
        user_id: The id of the user to add the items to the shopping cart.
        cart_id: The id of the shopping cart to add the items to.
        
    Returns:
        A list of the items added to the shopping cart.
    """

    conn = psycopg2.connect(
        host="aws-0-us-east-2.pooler.supabase.com",
        port=5432,
        database="postgres",
        user="tools_user.hhfpavgzueplacewdznr",
        password="tools_user_password"
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        
        for item in items:
            product_id = item['product_id']
            quantity = item['quantity']

            qdrant_client = QdrantClient(
                url="https://a7cde2b9-41b9-414c-9258-57aae924e200.us-west-1-0.aws.cloud.qdrant.io",
                api_key=os.getenv("QDRANT_API_KEY"),
                timeout=60 #include this so we do not get timeout errors when we upsert pointstructs/payload.vectors
            )

            dummy_vector = np.zeros(1536).tolist()
            payload = qdrant_client.query_points(
                collection_name="Amazon-items-collection-01-hybrid-search",
                prefetch=[
                    Prefetch(
                        query=dummy_vector,
                        filter=Filter(
                        must=[
                            FieldCondition(
                                key="parent_asin",
                                match=MatchValue(value=product_id)
                            )
                        ]
                    ),
                        using="text-embedding-3-small",
                        limit=20
                    )
                ],
                query=FusionQuery(fusion="rrf"),
                limit=1,
            ).points[0].payload

            product_image_url = payload.get("image")
            price = payload.get("price")
            currency = 'USD'
        
            # Check if item already exists
            check_query = """
                SELECT id, quantity, price 
                FROM shopping_carts.shopping_cart_items 
                WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
            """
            cursor.execute(check_query, (user_id, cart_id, product_id))
            existing_item = cursor.fetchone()
            
            if existing_item:
                # Update existing item
                new_quantity = existing_item['quantity'] + quantity
                
                update_query = """
                    UPDATE shopping_carts.shopping_cart_items 
                    SET 
                        quantity = %s,
                        price = %s,
                        currency = %s,
                        product_image_url = COALESCE(%s, product_image_url)
                    WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
                    RETURNING id, quantity, price
                """
                
                cursor.execute(update_query, (new_quantity, price, currency, product_image_url, user_id, cart_id, product_id))
            
            else:
                # Insert new item
                insert_query = """
                    INSERT INTO shopping_carts.shopping_cart_items (
                        user_id, shopping_cart_id, product_id,
                        price, quantity, currency, product_image_url
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s)
                    RETURNING id, quantity, price
                """
                
                cursor.execute(insert_query, (user_id, cart_id, product_id, price, quantity, currency, product_image_url))
            
    return f"Added {items} to the shopping cart."

In [5]:
add_to_shopping_cart(items, "user_1", "cart_1")

"Added [{'product_id': 'B0BR8SJM7N', 'quantity': 2}, {'product_id': 'B09JS5PSV1', 'quantity': 4}] to the shopping cart."

In [6]:
items_2 = [
    {
        "product_id": "B0BR8SJM7N",
        "quantity": 4
    }
]

In [7]:
add_to_shopping_cart(items_2, "user_1", "cart_1")

"Added [{'product_id': 'B0BR8SJM7N', 'quantity': 4}] to the shopping cart."

In [8]:
items_3 = [
    {
        "product_id": "B09X1LDMH6",
        "quantity": 1
    }
]

In [9]:
add_to_shopping_cart(items_3, "user_1", "cart_1")

"Added [{'product_id': 'B09X1LDMH6', 'quantity': 1}] to the shopping cart."

#### Get the shopping cart items tool

In [10]:
def get_shopping_cart(user_id: str, cart_id: str) -> list[dict]:

    """
    Retrieve all items in a user's shopping cart.
    
    Args:
        user_id: User identifier
        cart_id: Cart identifier
    
    Returns:
        List of dictionaries containing cart items
    """
    
    conn = psycopg2.connect(
        host="aws-0-us-east-2.pooler.supabase.com",
        port=5432,
        database="postgres",
        user="tools_user.hhfpavgzueplacewdznr",
        password="tools_user_password"
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:

        query = """
                SELECT 
                    product_id, price, quantity,
                    currency, product_image_url,
                    (price * quantity) as total_price
                FROM shopping_carts.shopping_cart_items 
                WHERE user_id = %s AND shopping_cart_id = %s
                ORDER BY added_at DESC
            """
        cursor.execute(query, (user_id, cart_id))

        return [dict(row) for row in cursor.fetchall()]

In [11]:
get_shopping_cart("user_1", "cart_1")

[{'product_id': 'B09X1LDMH6',
  'price': None,
  'quantity': 1,
  'currency': 'USD',
  'product_image_url': 'https://m.media-amazon.com/images/I/4192v6F66GL._AC_.jpg',
  'total_price': None},
 {'product_id': 'B09JS5PSV1',
  'price': Decimal('9.99'),
  'quantity': 4,
  'currency': 'USD',
  'product_image_url': 'https://m.media-amazon.com/images/I/41xqisIgiIL._AC_.jpg',
  'total_price': Decimal('39.96')},
 {'product_id': 'B0BR8SJM7N',
  'price': Decimal('9.99'),
  'quantity': 6,
  'currency': 'USD',
  'product_image_url': 'https://m.media-amazon.com/images/I/41H4croPOWL.jpg',
  'total_price': Decimal('59.94')}]

In [12]:
get_shopping_cart("user_1_asfadsf", "cart_1_asfasf")

[]

#### Deleting items from the Shopping Cart Tool

In [18]:
def remove_from_cart(product_id: str, user_id: str, cart_id: str) -> str:

    """
    Remove an item completely from the shopping cart.
    
    Args:
        user_id: User identifier
        product_id: Product identifier to remove
        cart_id: Cart identifier
    
    Returns:
        Information about the removal of the item from the shopping cart.
    """
    
    conn = psycopg2.connect(
        host="aws-0-us-east-2.pooler.supabase.com",
        port=5432,
        database="postgres",
        user="tools_user.hhfpavgzueplacewdznr",
        password="tools_user_password"
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:

        query = """
                DELETE FROM shopping_carts.shopping_cart_items
                WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
            """
        cursor.execute(query, (user_id, cart_id, product_id))

        return f"Removed {product_id} from the shopping cart." if cursor.rowcount > 0 else f"Item {product_id} not found in the shopping cart."

In [19]:
remove_from_cart("B09X1LDMH6", "user_1", "cart_1")

'Removed B09X1LDMH6 from the shopping cart.'

In [20]:
remove_from_cart("B09JS5PSV1", "user_1", "cart_1")

'Removed B09JS5PSV1 from the shopping cart.'

In [21]:
add_to_shopping_cart(items, "user_1", "cart_1")

"Added [{'product_id': 'B0BR8SJM7N', 'quantity': 2}, {'product_id': 'B09JS5PSV1', 'quantity': 4}] to the shopping cart."

In [22]:
get_shopping_cart("user_1", "cart_1")

[{'product_id': 'B09JS5PSV1',
  'price': Decimal('9.99'),
  'quantity': 4,
  'currency': 'USD',
  'product_image_url': 'https://m.media-amazon.com/images/I/41xqisIgiIL._AC_.jpg',
  'total_price': Decimal('39.96')},
 {'product_id': 'B0BR8SJM7N',
  'price': Decimal('9.99'),
  'quantity': 8,
  'currency': 'USD',
  'product_image_url': 'https://m.media-amazon.com/images/I/41H4croPOWL.jpg',
  'total_price': Decimal('79.92')}]

In [23]:
add_to_shopping_cart(items, "user_1", "cart_1")

"Added [{'product_id': 'B0BR8SJM7N', 'quantity': 2}, {'product_id': 'B09JS5PSV1', 'quantity': 4}] to the shopping cart."

In [24]:
get_shopping_cart("user_1", "cart_1")

[{'product_id': 'B09JS5PSV1',
  'price': Decimal('9.99'),
  'quantity': 8,
  'currency': 'USD',
  'product_image_url': 'https://m.media-amazon.com/images/I/41xqisIgiIL._AC_.jpg',
  'total_price': Decimal('79.92')},
 {'product_id': 'B0BR8SJM7N',
  'price': Decimal('9.99'),
  'quantity': 10,
  'currency': 'USD',
  'product_image_url': 'https://m.media-amazon.com/images/I/41H4croPOWL.jpg',
  'total_price': Decimal('99.90')}]

In [25]:
remove_from_cart("B0BR8SJM7N", "user_1", "cart_1")

'Removed B0BR8SJM7N from the shopping cart.'

In [26]:
get_shopping_cart("user_1", "cart_1")

[{'product_id': 'B09JS5PSV1',
  'price': Decimal('9.99'),
  'quantity': 8,
  'currency': 'USD',
  'product_image_url': 'https://m.media-amazon.com/images/I/41xqisIgiIL._AC_.jpg',
  'total_price': Decimal('79.92')}]

### Warehouse Manager Agent Tools

#### Simulate availability in warehouses

#### Import Dependencies

In [27]:
import psycopg2
from psycopg2.extras import RealDictCursor, execute_batch
import numpy as np
from qdrant_client import QdrantClient
import random

### Fictional Warehouses

In [30]:
warehouses = [
    {
        "warehouse_id": "DE-BER-01",
        "warehouse_location": "Berlin, Germany",
        "warehouse_name": "Berlin Distribution Center"
    },
    {
        "warehouse_id": "DE-MUN-01",
        "warehouse_location": "Munich, Germany",
        "warehouse_name": "Munich Logistics Hub"
    },
    {
        "warehouse_id": "DE-HAM-01",
        "warehouse_location": "Hamburg, Germany",
        "warehouse_name": "Hamburg North Warehouse"
    },
    {
        "warehouse_id": "FR-PAR-01",
        "warehouse_location": "Paris, France",
        "warehouse_name": "Paris Central Depot"
    },
    {
        "warehouse_id": "FR-LYO-01",
        "warehouse_location": "Lyon, France",
        "warehouse_name": "Lyon Regional Warehouse"
    },
    {
        "warehouse_id": "FR-MAR-01",
        "warehouse_location": "Marseille, France",
        "warehouse_name": "Marseille Mediterranean Hub"
    }
]

##### Simulate Stock Availability for each of the warehouse

##### Retrieve all item IDs from the Amzn items Qdrant collection

In [31]:
qdrant_client = QdrantClient(
                url="https://a7cde2b9-41b9-414c-9258-57aae924e200.us-west-1-0.aws.cloud.qdrant.io",
                api_key=os.getenv("QDRANT_API_KEY"),
                timeout=60 #include this so we do not get timeout errors 
            )

In [32]:
dummy_vector = np.zeros(1536).tolist()

In [33]:
payload = qdrant_client.query_points(
    collection_name="Amazon-items-collection-01-hybrid-search",
    query=dummy_vector,
    using="text-embedding-3-small",
    limit=1000,
    with_payload=["parent_asin"],
    with_vectors=False
)

In [34]:
payload.points

[ScoredPoint(id=52, version=3, score=0.0, payload={'parent_asin': 'B0BRPWQWPN'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=380, version=3, score=0.0, payload={'parent_asin': 'B0C5WGFL2N'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=390, version=3, score=0.0, payload={'parent_asin': 'B09SHMQ5MK'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=463, version=3, score=0.0, payload={'parent_asin': 'B0BW5KSV6Z'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=426, version=3, score=0.0, payload={'parent_asin': 'B0CH8F91CP'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=59, version=3, score=0.0, payload={'parent_asin': 'B09LTX3SQX'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=143, version=3, score=0.0, payload={'parent_asin': 'B0BJDTBLWL'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=149, version=3, score=0.0, payload={'parent_asin': 'B09S6HJGBH'}, vector=Non

In [35]:
len(payload.points)

1000

In [36]:
parent_asin_list = [item.payload["parent_asin"] for item in payload.points]

In [37]:
parent_asin_list

['B0BRPWQWPN',
 'B0C5WGFL2N',
 'B09SHMQ5MK',
 'B0BW5KSV6Z',
 'B0CH8F91CP',
 'B09LTX3SQX',
 'B0BJDTBLWL',
 'B09S6HJGBH',
 'B09533Z3XY',
 'B09ZTV5K7G',
 'B0973FM88X',
 'B09YTGFYZK',
 'B09VKMVF2F',
 'B09PT6JFYW',
 'B0B5HVSQ2L',
 'B0B1579XMT',
 'B0BWM11RM6',
 'B0BGC9N61V',
 'B0C48TYFDN',
 'B0B5KVF7YL',
 'B0BCHTW9PC',
 'B0BWRHJR69',
 'B0B51SZ6J9',
 'B0BYRXT52V',
 'B0B5K9NFHV',
 'B0BSD3QK7M',
 'B0B5LSJWJ6',
 'B09W9JNG7B',
 'B0B1QVXFD8',
 'B0C996WY16',
 'B09MM5QT3R',
 'B0BK8WW5DH',
 'B0BVH2C84P',
 'B0B97MYSHW',
 'B0BC4PGXFK',
 'B0C28D8532',
 'B0C6MJL8D8',
 'B0B5WTFT3X',
 'B0B4915S57',
 'B0BQGDJZV8',
 'B09STXZRMS',
 'B0BZ44Y4C6',
 'B09PP95VQ5',
 'B0B2WL2MTC',
 'B09X9RM6FD',
 'B0C22LCNVV',
 'B09WDMB54W',
 'B0BGH3H1WM',
 'B0B7QSQ1SJ',
 'B0B66H8W3K',
 'B09S3GRY4Q',
 'B09VBWBHBC',
 'B09S5H3VGH',
 'B0B87L83J7',
 'B0B2J5JHC8',
 'B0C6GQ4YQT',
 'B0B63FT91X',
 'B0B1MNQGT9',
 'B089LDYYPM',
 'B09TBF91FL',
 'B0B2KJKT86',
 'B09W9FDD1X',
 'B0BHKS1W55',
 'B0BTVK6NX1',
 'B0BVZ5R2VN',
 'B097J4V43J',
 'B0BRN967

##### Generate synthetic availability for all items in Qdrant

In [38]:
def generate_inventory_data(warehouses, product_ids, availability_rate=0.75):
    
    inventory_records = []
    
    for warehouse in warehouses:
        for product_id in product_ids:
            # 75% chance the product is available in this warehouse
            if random.random() < availability_rate:
                total_quantity = random.randint(0, 100)
                
                # Only add to inventory if quantity > 0
                if total_quantity > 0:
                    inventory_records.append({
                        "warehouse_id": warehouse["warehouse_id"],
                        "warehouse_location": warehouse["warehouse_location"],
                        "warehouse_name": warehouse["warehouse_name"],
                        "product_id": product_id,
                        "total_quantity": total_quantity,
                        "reserved_quantity": 0  # Starting with no reservations
                    })
    
    return inventory_records

In [39]:
inventory_data = generate_inventory_data(warehouses, parent_asin_list, availability_rate=0.75)

In [40]:
inventory_data

[{'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B0BRPWQWPN',
  'total_quantity': 43,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B0C5WGFL2N',
  'total_quantity': 32,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B0BW5KSV6Z',
  'total_quantity': 87,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B0CH8F91CP',
  'total_quantity': 16,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B09LTX3SQX',
  'total_quantity': 26,
  

In [41]:
1000*0.75*6

4500.0

In [42]:
len(inventory_data)

4431

##### Write synthetic data to Postgres

In [43]:
def insert_inventory_to_db(inventory_records):
   
    try:
        # Connect to the database
        conn = psycopg2.connect(
        host="aws-0-us-east-2.pooler.supabase.com",
        port=5432,
        database="postgres",
        user="tools_user.hhfpavgzueplacewdznr",
        password="tools_user_password"
        )
        conn.autocommit = True

        with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        
            # Prepare the INSERT query
            insert_query = """
            INSERT INTO warehouses.inventory 
            (warehouse_id, warehouse_location, warehouse_name, product_id, total_quantity, reserved_quantity)
            VALUES (%(warehouse_id)s, %(warehouse_location)s, %(warehouse_name)s, %(product_id)s, %(total_quantity)s, %(reserved_quantity)s)
            """
            
            # Use execute_batch for better performance with many inserts
            execute_batch(cursor, insert_query, inventory_records, page_size=100)
            
            # Commit the transaction
            conn.commit()
            
            print(f"Successfully inserted {len(inventory_records)} records into warehouses.inventory")
            
            # Close cursor and connection
            cursor.close()
            conn.close()
        
    except psycopg2.Error as e:
        print(f"Database error: {e}")
        if conn:
            conn.rollback()
    except Exception as e:
        print(f"Error: {e}")
    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()

In [44]:
insert_inventory_to_db(inventory_data)

Successfully inserted 4431 records into warehouses.inventory


#### Test Warehouse Agent Tools

##### Item Availability Check Tool

In [45]:
def check_warehouse_availability(items: list[dict]) -> dict:

    """Check availability of items across warehouses, including partial fulfillment options.
    
    Args:
        items: A list of items to check. Each item is a dictionary with keys: product_id, quantity.
        
    Returns:
        A dictionary containing:
        - can_fulfill_completely: bool indicating if all items can be fulfilled from at least one warehouse
        - warehouses_full_fulfillment: list of warehouses that can fulfill the entire order
        - warehouses_partial_fulfillment: list of warehouses with partial availability
        - unavailable_items: list of items that cannot be fulfilled from any warehouse
        - details: detailed breakdown per warehouse with availability for each item
    """
    conn = psycopg2.connect(
        host="aws-0-us-east-2.pooler.supabase.com",
        port=5432,
        database="postgres",
        user="tools_user.hhfpavgzueplacewdznr",
        password="tools_user_password"
    )
    
    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cursor:
            result = {
                "can_fulfill_completely": False,
                "warehouses_full_fulfillment": [],
                "warehouses_partial_fulfillment": [],
                "unavailable_items": [],
                "details": []
            }
            
            # Check each warehouse for availability
            warehouse_query = """
                SELECT DISTINCT warehouse_id, warehouse_name, warehouse_location
                FROM warehouses.inventory
            """
            cursor.execute(warehouse_query)
            warehouses = cursor.fetchall()
            
            for warehouse in warehouses:
                warehouse_can_fulfill_all = True
                has_any_availability = False
                warehouse_details = {
                    "warehouse_id": warehouse['warehouse_id'],
                    "warehouse_name": warehouse['warehouse_name'],
                    "warehouse_location": warehouse['warehouse_location'],
                    "items": [],
                    "can_fulfill_all": False,
                    "has_partial": False
                }
                
                for item in items:
                    product_id = item['product_id']
                    requested_quantity = item['quantity']
                    
                    # Check availability in this warehouse
                    availability_query = """
                        SELECT product_id, total_quantity, reserved_quantity, available_quantity
                        FROM warehouses.inventory
                        WHERE warehouse_id = %s AND product_id = %s
                    """
                    cursor.execute(availability_query, (warehouse['warehouse_id'], product_id))
                    inventory = cursor.fetchone()
                    
                    available_qty = inventory['available_quantity'] if inventory else 0
                    
                    item_detail = {
                        "product_id": product_id,
                        "requested": requested_quantity,
                        "available": available_qty,
                        "can_fulfill_completely": available_qty >= requested_quantity,
                        "can_fulfill_partially": available_qty > 0 and available_qty < requested_quantity
                    }
                    
                    warehouse_details["items"].append(item_detail)
                    
                    # Track if warehouse can fulfill this item completely
                    if available_qty < requested_quantity:
                        warehouse_can_fulfill_all = False
                    
                    # Track if warehouse has any availability for any item
                    if available_qty > 0:
                        has_any_availability = True
                
                # Categorize warehouse
                if warehouse_can_fulfill_all:
                    warehouse_details["can_fulfill_all"] = True
                    result["warehouses_full_fulfillment"].append({
                        "warehouse_id": warehouse['warehouse_id'],
                        "warehouse_name": warehouse['warehouse_name'],
                        "warehouse_location": warehouse['warehouse_location']
                    })
                elif has_any_availability:
                    warehouse_details["has_partial"] = True
                    result["warehouses_partial_fulfillment"].append({
                        "warehouse_id": warehouse['warehouse_id'],
                        "warehouse_name": warehouse['warehouse_name'],
                        "warehouse_location": warehouse['warehouse_location']
                    })
                
                result["details"].append(warehouse_details)
            
            # Check if any items cannot be fulfilled from any warehouse
            for item in items:
                product_id = item['product_id']
                requested_quantity = item['quantity']
                
                # Get total available quantity across all warehouses
                total_available_query = """
                    SELECT product_id, SUM(available_quantity) as total_available
                    FROM warehouses.inventory
                    WHERE product_id = %s
                    GROUP BY product_id
                """
                cursor.execute(total_available_query, (product_id,))
                total_available = cursor.fetchone()
                
                total_available_qty = total_available['total_available'] if total_available else 0
                
                if total_available_qty < requested_quantity:
                    result["unavailable_items"].append({
                        "product_id": product_id,
                        "requested": requested_quantity,
                        "total_available_across_warehouses": total_available_qty,
                        "shortage": requested_quantity - total_available_qty
                    })
            
            result["can_fulfill_completely"] = len(result["warehouses_full_fulfillment"]) > 0 and len(result["unavailable_items"]) == 0
            
            return result
            
    finally:
        conn.close()

In [46]:
shopping_cart = [
    {
        "product_id": "B0C747Q3K1",
        "quantity": 3
    },
    {
        "product_id": "B09P5NQH7Q",
        "quantity": 2
    },
    {
        "product_id": "B0CFLR6R3X",
        "quantity": 5
    }
]

In [47]:
availability = check_warehouse_availability(shopping_cart)

In [48]:
availability

{'can_fulfill_completely': False,
 'warehouses_full_fulfillment': [],
 'warehouses_partial_fulfillment': [{'warehouse_id': 'DE-BER-01',
   'warehouse_name': 'Berlin Distribution Center',
   'warehouse_location': 'Berlin, Germany'},
  {'warehouse_id': 'FR-LYO-01',
   'warehouse_name': 'Lyon Regional Warehouse',
   'warehouse_location': 'Lyon, France'},
  {'warehouse_id': 'DE-MUN-01',
   'warehouse_name': 'Munich Logistics Hub',
   'warehouse_location': 'Munich, Germany'},
  {'warehouse_id': 'FR-MAR-01',
   'warehouse_name': 'Marseille Mediterranean Hub',
   'warehouse_location': 'Marseille, France'},
  {'warehouse_id': 'DE-HAM-01',
   'warehouse_name': 'Hamburg North Warehouse',
   'warehouse_location': 'Hamburg, Germany'}],
 'unavailable_items': [],
 'details': [{'warehouse_id': 'DE-BER-01',
   'warehouse_name': 'Berlin Distribution Center',
   'warehouse_location': 'Berlin, Germany',
   'items': [{'product_id': 'B0C747Q3K1',
     'requested': 3,
     'available': 43,
     'can_fulfill

#### Item Reservation tool

In [49]:
def reserve_warehouse_items(reservations: list[dict]) -> dict:
    
    """Reserve items from multiple warehouses in a single transaction.
    
    Args:
        reservations: A list of reservations. Each reservation is a dictionary with keys:
                     - warehouse_id: The warehouse to reserve from
                     - product_id: The product to reserve
                     - quantity: The quantity to reserve
        
    Returns:
        A dictionary containing:
        - success: bool indicating if all reservations were successful
        - reserved_items: list of successfully reserved items
        - failed_items: list of items that could not be reserved
    """
    
    conn = psycopg2.connect(
        host="aws-0-us-east-2.pooler.supabase.com",
        port=5432,
        database="postgres",
        user="tools_user.hhfpavgzueplacewdznr",
        password="tools_user_password"
    )
    conn.autocommit = False  # Use transaction
    
    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cursor:
            result = {
                "success": False,
                "reserved_items": [],
                "failed_items": []
            }
            
            for reservation in reservations:
                warehouse_id = reservation['warehouse_id']
                product_id = reservation['product_id']
                quantity = reservation['quantity']
                
                # Check and lock the inventory row
                check_query = """
                    SELECT warehouse_id, product_id, warehouse_name, warehouse_location, 
                           total_quantity, reserved_quantity, available_quantity
                    FROM warehouses.inventory
                    WHERE warehouse_id = %s AND product_id = %s
                    FOR UPDATE
                """
                cursor.execute(check_query, (warehouse_id, product_id))
                inventory = cursor.fetchone()
                
                if inventory and inventory['available_quantity'] >= quantity:
                    # Update inventory to reserve the items
                    update_query = """
                        UPDATE warehouses.inventory
                        SET reserved_quantity = reserved_quantity + %s
                        WHERE warehouse_id = %s AND product_id = %s
                    """
                    cursor.execute(update_query, (quantity, warehouse_id, product_id))
                    
                    result["reserved_items"].append({
                        "product_id": product_id,
                        "quantity": quantity,
                        "warehouse_id": warehouse_id,
                        "warehouse_name": inventory['warehouse_name'],
                        "warehouse_location": inventory['warehouse_location']
                    })
                else:
                    result["failed_items"].append({
                        "product_id": product_id,
                        "warehouse_id": warehouse_id,
                        "requested": quantity,
                        "available": inventory['available_quantity'] if inventory else 0,
                        "reason": "insufficient_stock" if inventory else "not_in_warehouse"
                    })
            
            # Only commit if all items were successfully reserved
            if len(result["failed_items"]) == 0:
                conn.commit()
                result["success"] = True
            else:
                conn.rollback()
                result["success"] = False
            
            return result
            
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        conn.close()

In [54]:
shopping_cart = [
    {
        "product_id": "B0C747Q3K1",
        "quantity": 3,
        "warehouse_id": 'DE-BER-01'
    },
    {
        "product_id": "B09P5NQH7Q",
        "quantity": 2,
        "warehouse_id": "FR-MAR-01"
    },
    {
        "product_id": "B0CFLR6R3X",
        "quantity": 5,
        "warehouse_id": "DE-BER-01"
    }
]

In [55]:
reserve_warehouse_items(shopping_cart)

{'success': True,
 'reserved_items': [{'product_id': 'B0C747Q3K1',
   'quantity': 3,
   'warehouse_id': 'DE-BER-01',
   'warehouse_name': 'Berlin Distribution Center',
   'warehouse_location': 'Berlin, Germany'},
  {'product_id': 'B09P5NQH7Q',
   'quantity': 2,
   'warehouse_id': 'FR-MAR-01',
   'warehouse_name': 'Marseille Mediterranean Hub',
   'warehouse_location': 'Marseille, France'},
  {'product_id': 'B0CFLR6R3X',
   'quantity': 5,
   'warehouse_id': 'DE-BER-01',
   'warehouse_name': 'Berlin Distribution Center',
   'warehouse_location': 'Berlin, Germany'}],
 'failed_items': []}

In [56]:
shopping_cart = [
    {
        "product_id": "B0C747Q3K1",
        "quantity": 120,
        "warehouse_id": "DE-BER-01"
    },
    {
        "product_id": "B09P5NQH7Q",
        "quantity": 2,
        "warehouse_id": "FR-LYO-01"
    },
    {
        "product_id": "B0CFLR6R3X",
        "quantity": 5,
        "warehouse_id": "FR-LYO-01"
    }
]

In [53]:
reserve_warehouse_items(shopping_cart)

{'success': False,
 'reserved_items': [{'product_id': 'B0CFLR6R3X',
   'quantity': 5,
   'warehouse_id': 'FR-LYO-01',
   'warehouse_name': 'Lyon Regional Warehouse',
   'warehouse_location': 'Lyon, France'}],
 'failed_items': [{'product_id': 'B0C747Q3K1',
   'warehouse_id': 'FR-LYO-01',
   'requested': 120,
   'available': 34,
   'reason': 'insufficient_stock'},
  {'product_id': 'B09P5NQH7Q',
   'warehouse_id': 'FR-LYO-01',
   'requested': 2,
   'available': 0,
   'reason': 'not_in_warehouse'}]}